<a href="https://colab.research.google.com/github/anuradha-gh/AI-Powered-Granular-Access-Control-for-SaaS-Applications-/blob/Anomaly-Detection/anomaly_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# @title 1. Setup and Installations
!pip install tensorflow scikit-learn pandas -q

import json
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
# We import IsolationForest but will not use it yet
from sklearn.ensemble import IsolationForest
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from google.colab import drive
import warnings

warnings.filterwarnings('ignore')
print("Libraries installed and imported.")

# @title 2. Load and Parse CloudTrail Log Data
# --- Configuration ---
# This notebook uses the AWS CloudTrail logs.
FILE_PATH = '/content/drive/MyDrive/flaws_cloudtrail00.json'
# ---------------------

try:
    drive.mount('/content/drive')
    with open(FILE_PATH, 'r') as f:
        log_data = json.load(f)
    records = log_data.get('Records', [])
    df = pd.json_normalize(records)
    print(f"Successfully loaded {len(df)} log records from {FILE_PATH}.")
except FileNotFoundError:
    print(f"ERROR: File '{FILE_PATH}' not found.")
    print("Please upload 'flaws_cloudtrail00.json' to your Google Drive and update the path.")
    df = pd.DataFrame()


# @title 3. Feature Engineering (Initial: One-Hot Encoding)
if not df.empty:
    # For this baseline, we select simple categorical features.

    # 1. Fill missing values
    df['eventName'] = df['eventName'].fillna('Unknown')
    df['eventSource'] = df['eventSource'].fillna('Unknown')
    df['awsRegion'] = df['awsRegion'].fillna('Unknown')
    df['userIdentity.type'] = df['userIdentity.type'].fillna('Unknown')

    # 2. Define features for preprocessing
    categorical_features = ['eventName', 'eventSource', 'awsRegion', 'userIdentity.type']

    # Use top 20 most frequent categories
    for col in categorical_features:
        top_20 = df[col].value_counts().nlargest(20).index
        df[col] = df[col].apply(lambda x: x if x in top_20 else 'Other')

    # 3. Create the preprocessing pipeline
    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
        ],
        remainder='drop'
    )

    print("Preprocessing pipeline (OneHotEncoder) created.")

    # 4. Apply preprocessing
    X_processed = preprocessor.fit_transform(df)
    X_processed = X_processed.toarray()

    print(f"Data processed. Feature shape: {X_processed.shape}")


# @title 4. Build and Train Autoencoder
if 'X_processed' in locals():
    # Define the Autoencoder architecture
    input_dim = X_processed.shape[1]
    encoding_dim = 16  # Latent vector dimension

    input_layer = Input(shape=(input_dim,))
    encoder = Dense(64, activation='relu')(input_layer)
    encoder = Dense(32, activation='relu')(encoder)
    encoder = Dense(encoding_dim, activation='relu')(encoder)

    decoder = Dense(32, activation='relu')(encoder)
    decoder = Dense(64, activation='relu')(decoder)
    decoder = Dense(input_dim, activation='sigmoid')(decoder)

    autoencoder = Model(inputs=input_layer, outputs=decoder)
    autoencoder.compile(optimizer='adam', loss='mean_squared_error')

    print("Autoencoder model compiled.")
    autoencoder.summary()

    # Train the Autoencoder
    print("\nTraining Autoencoder to learn 'normal' behavior...")
    # Using only 5 epochs to show a quick "initial" training pass
    autoencoder.fit(
        X_processed,
        X_processed,
        epochs=5, # Reduced epochs for initial demo
        batch_size=32,
        shuffle=True,
        validation_split=0.1,
        verbose=1
    )
    print("Initial Autoencoder training complete.")


# @title 5. Show Results and Next Steps
if 'autoencoder' in locals():
    print(f"\n--- Component 1: Initial Stage Demo ---")

    # Get reconstruction error for the first 5 samples as a demo
    X_sample = X_processed[:5]
    X_reconstructed_sample = autoencoder.predict(X_sample)
    mse_sample = np.mean(np.power(X_sample - X_reconstructed_sample, 2), axis=1)

    print("\n--- Example Reconstruction Errors (MSE) ---")
    for i, mse in enumerate(mse_sample):
      print(f"  Log Entry {i}: MSE = {mse:.6f}")


